In [37]:
import pandas as pd
import numpy as np

## 1. Correção de tipos de dados
As colunas de data foram lidas como string pelo Pandas.
Convertendo para datetime para permitir cálculos de diferença de datas.

In [38]:
df_pedidos = pd.read_csv('../data/raw/olist_order_items_dataset.csv')

In [39]:
df_pedidos['shipping_limit_date'] = pd.to_datetime(df_pedidos['shipping_limit_date'])

In [40]:
df_pedidos.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

In [41]:
df_avaliacoes = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

In [42]:
df_avaliacoes['review_creation_date'] = pd.to_datetime(df_avaliacoes['review_creation_date'])

In [43]:
df_avaliacoes.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp               str
dtype: object

In [44]:
df_compras = pd.read_csv('../data/raw/olist_orders_dataset.csv')

In [45]:
df_compras[['order_purchase_timestamp', 'order_approved_at','order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']] = df_compras[['order_purchase_timestamp', 'order_approved_at','order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']].apply(lambda x: pd.to_datetime(x, errors='coerce'))

In [46]:
df_compras.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 2. Tratamento de nulos — df_produtos
Removidas linhas onde todas as colunas (exceto product_id) estão nulas.
Eram registros fantasma sem informação utilizável. 1 linha removida de 32.951.

In [47]:
df_produtos = pd.read_csv('../data/raw/olist_products_dataset.csv')

In [48]:
df_produtos.shape

(32951, 9)

In [49]:
colunas_sem_id = [col for col in df_produtos.columns if col != 'product_id']

linhas_vazias = df_produtos[df_produtos[colunas_sem_id].isna().all(axis=1)]

df_produtos = df_produtos.dropna(subset=colunas_sem_id, how='all')

In [50]:
df_produtos.shape

(32950, 9)

## 3. Colunas calculadas
- `order_year` / `order_month`: extraídos para análise de sazonalidade
- `status_simplificado`: agrupamento de 8 status em 3 categorias de negócio


In [51]:
status_simplificado = {
    'delivered': 'Concluído',
    'shipped': 'Em Andamento',
    'invoiced': 'Em Andamento',
    'processing': 'Em Andamento',
    'approved': 'Em Andamento',
    'created': 'Em Andamento',
    'canceled': 'Cancelado',
    'unavailable': 'Cancelado'
}


df_compras = df_compras.copy()

df_compras['status_simplificado'] = df_compras['order_status'].map(status_simplificado)

df_compras['order_year'] = df_compras['order_purchase_timestamp'].dt.year
df_compras['order_month'] = df_compras['order_purchase_timestamp'].dt.month

status_nulos = df_compras['status_simplificado'].isna().sum()
if status_nulos > 0:
    print(f" Atenção: Existem {status_nulos} linhas com status não mapeados no dicionário!")
else:
    print("Todos os status foram simplificados com sucesso.")





Todos os status foram simplificados com sucesso.


In [52]:
df_compras.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,status_simplificado,order_year,order_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,Concluído,2017,10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,Concluído,2018,7
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,Concluído,2018,8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,Concluído,2017,11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,Concluído,2018,2


In [53]:
df_pagamento = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

df_pagamento['faixa_valor'] = pd.qcut(df_pagamento['payment_value'], q=10, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10'])

df_pagamento.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value,faixa_valor
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,Q5
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,Q1
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,Q4
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,Q6
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,Q7


In [54]:
df_consumidores = pd.read_csv('../data/raw/olist_customers_dataset.csv')
df_traducao = pd.read_csv('../data/raw/product_category_name_translation.csv')
df_vendedor = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

## 4. Consolidação dos dados — Merge entre tabelas

Para análise, as informações de clientes, pagamentos, itens, vendedores e avaliações
estão em tabelas separadas. Antes do merge, pagamentos e itens foram agregados por
`order_id` para evitar duplicação de linhas — um pedido pode ter múltiplos itens e
mais de uma forma de pagamento.

A tabela `df_compras` é usada como base. Todos os joins são `LEFT JOIN` para garantir
que nenhum pedido seja perdido, mesmo que falte informação em alguma tabela auxiliar.

**Resultado:** `df_geral` com 99441 linhas e 26 colunas, mesmo número de linhas que `df_compras`.


In [55]:
# ── 1. Agregar avaliações por pedido ────────────────
df_avaliacoes_agg = df_avaliacoes.groupby('order_id').agg(
    review_score=('review_score', 'mean')
).reset_index()

# ── 2. Agregar pagamentos por pedido ────────────────
df_pagamento_agg = df_pagamento.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', lambda x: x.mode()[0]),
    payment_installments=('payment_installments', 'max')
).reset_index()

# ── 3. Agregar itens por pedido ─────────────────────
df_pedidos_agg = df_pedidos.groupby('order_id').agg(
    price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    quantidade_itens=('order_item_id', 'count'),
    seller_id=('seller_id', 'first')
).reset_index()

# ── 4. Traduzir categorias ──────────────────────────
df_produtos = df_produtos.merge(df_traducao, on='product_category_name', how='left')

# ── 4. Merge em sequência ───────────────────────────
df_geral = df_compras \
    .merge(df_consumidores,         on='customer_id', how='left') \
    .merge(df_pagamento_agg,    on='order_id',    how='left') \
    .merge(df_pedidos_agg,      on='order_id',    how='left') \
    .merge(df_vendedor,         on='seller_id',   how='left') \
    .merge(df_avaliacoes_agg,   on='order_id',    how='left')

# ── 5. Verificar resultado ──────────────────────────
print(f"Linhas em df_compras:  {len(df_compras)}")
print(f"Linhas em df_geral:    {len(df_geral)}")
print(f"Colunas: {df_geral.columns.tolist()}")

Linhas em df_compras:  99441
Linhas em df_geral:    99441
Colunas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'status_simplificado', 'order_year', 'order_month', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_value', 'payment_type', 'payment_installments', 'price', 'freight_value', 'quantidade_itens', 'seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'review_score']


In [56]:
df_geral.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,status_simplificado,order_year,...,payment_type,payment_installments,price,freight_value,quantidade_itens,seller_id,seller_zip_code_prefix,seller_city,seller_state,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,Concluído,2017,...,voucher,1.0,29.99,8.72,1.0,3504c0cb71d7fa48d967e0e4c94d59d9,9350.0,maua,SP,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,Concluído,2018,...,boleto,1.0,118.70,22.76,1.0,289cdb325fb7e7f891c38608bf9e0962,31570.0,belo horizonte,SP,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,Concluído,2018,...,credit_card,3.0,159.90,19.22,1.0,4869f7a5dfa277a7dca6462dcf3b52b2,14840.0,guariba,SP,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,Concluído,2017,...,credit_card,1.0,45.00,27.20,1.0,66922902710d126a0e7d26b0e3805106,31842.0,belo horizonte,MG,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,Concluído,2018,...,credit_card,1.0,19.90,8.72,1.0,2c9e548be18521d1c43cde1c582c6de8,8752.0,mogi das cruzes,SP,5.0


## 5. Criação de df_compras_entregues e cálculo de tempo de entrega

`df_compras_entregues` é um subconjunto de `df_geral` contendo apenas pedidos com
`order_status == 'delivered'`. Pedidos cancelados, em andamento ou com outros status
são mantidos no `df_geral` para análises de cancelamento e volume geral.

A coluna `time_delivered` representa o tempo total que o cliente esperou pelo pedido,
calculado entre `order_purchase_timestamp` (momento da compra) e
`order_delivered_customer_date` (entrega ao cliente). O valor é arredondado para cima
com `np.ceil` porque do ponto de vista do cliente, 8.1 dias de espera são 9 dias.

Linhas com datas nulas foram removidas antes do cálculo — são pedidos marcados como
entregues mas sem data de entrega registrada, o que indica inconsistência no dado.

In [57]:
df_compras_entregues = df_geral[df_geral['order_status'] == 'delivered'].copy()

df_compras_entregues.dropna(
    subset=['order_delivered_customer_date', 'order_purchase_timestamp'],
    inplace=True
)

df_compras_entregues['time_delivered'] = np.ceil(
    (df_compras_entregues['order_delivered_customer_date'] -
     df_compras_entregues['order_purchase_timestamp']) / pd.Timedelta(days=1)
).astype(int)

In [58]:
print("=== df_geral ===")
print(f"Linhas: {len(df_geral)} | Colunas: {len(df_geral.columns)}")
print(f"Nulos por coluna:\n{df_geral.isnull().sum()[df_geral.isnull().sum() > 0]}")

print("\n=== df_compras_entregues ===")
print(f"Linhas: {len(df_compras_entregues)} | Colunas: {len(df_compras_entregues.columns)}")
print(f"time_delivered — min: {df_compras_entregues['time_delivered'].min()} | max: {df_compras_entregues['time_delivered'].max()}")

=== df_geral ===
Linhas: 99441 | Colunas: 26
Nulos por coluna:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
payment_value                       1
payment_type                        1
payment_installments                1
price                             775
freight_value                     775
quantidade_itens                  775
seller_id                         775
seller_zip_code_prefix            775
seller_city                       775
seller_state                      775
review_score                      768
dtype: int64

=== df_compras_entregues ===
Linhas: 96470 | Colunas: 27
time_delivered — min: 1 | max: 210


## 6. Salvando as novas planilhas
-`orders_full`: registro de todas as compras  
-`orders_delivered`: registro apenas das compras que tem o status de entregue com a data da compra e data da entrega.

In [59]:
df_geral.to_csv('../data/processed/orders_full.csv', index=False)
df_compras_entregues.to_csv('../data/processed/orders_delivered.csv', index=False)